# residual-skip-add — worked example 3: Skip-add keeps gradients flowing through the input

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `residual-skip-add`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The additive skip connection means the input appears directly in the output sum, so during backprop the gradient at the input is at least the upstream gradient passed straight through the identity branch (plus whatever flows through `F`). This is why residual networks avoid vanishing gradients in deep stacks.

## Worked solution

We build an identity-shortcut block whose residual function is a single 1x1 conv (channel-preserving). After a forward pass on a leaf input with `requires_grad`, we backpropagate a ones gradient and inspect `x.grad`. Because `out = conv(x) + x`, the gradient of the sum with respect to `x` includes a direct `+1` contribution from the identity branch, so `x.grad` is non-trivial everywhere. We print the mean absolute input gradient to confirm gradients reach the input through the skip path.

In [ ]:
import torch.nn as nn
Tensor = t.Tensor


class IdResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.f = nn.Conv2d(channels, channels, kernel_size=1)
        self.skip = nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
block = IdResBlock(4)
x = t.randn(1, 4, 8, 8, requires_grad=True)
out = block(x)
out.sum().backward()
print('input grad is populated:', x.grad is not None)
print('mean abs input grad:', round(x.grad.abs().mean().item(), 4))